# M1 error analysis

> **SAMPLE / SYNTHETIC DATA.** The declarations analysed here were written by an AI model (blind to the
> performance-criterion text) to test the pipeline. They are **not** real workers. Re-run this notebook
> on `experiments/m1_predictions_real.csv` once the team-written set exists.

Inputs: `experiments/m1_predictions_synthetic.csv` (one row per declaration and method, written by
`python -m hunar.mapper.evaluate`) and `data/synthetic/declarations_synthetic.csv` (for the difficulty tag).

In [1]:
import json, pandas as pd, numpy as np
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "experiments").exists() else Path.cwd().parent
pred = pd.read_csv(ROOT / "experiments/m1_predictions_synthetic.csv")
decl = pd.read_csv(ROOT / "data/synthetic/declarations_synthetic.csv")[["id", "difficulty"]]
pred = pred.merge(decl, on="id", how="left")
packs = {p["nqr_id"]: p["title"] for p in json.load(open(ROOT / "data/qp_sources.json"))["packs"]}
# pred_top1 is empty when every pack scored 0; those rows count as wrong (no NaN may be dropped from a mean)
pred["pred_top1"] = pd.to_numeric(pred["pred_top1"], errors="coerce")
pred["correct_top1"] = (pred["pred_top1"] == pred["gold"]).astype(bool)
print(pred.shape, "rows;", pred["method"].unique().tolist())

(750, 11) rows; ['keyword', 'tfidf', 'minilm', 'labse', 'hybrid']


## 1. Accuracy by method, language and difficulty

Crude top-1 from the arg-max (an all-zero score vector counts as wrong). The official metrics in
`experiments/m1_mapper_synthetic.json` score ties as an expected value, so they differ slightly.

In [2]:
pd.pivot_table(pred, index="method", columns=["language"], values="correct_top1", aggfunc="mean").round(2)

language,en,hi,hinglish
method,,,
hybrid,0.72,0.44,0.62
keyword,0.64,0.04,0.60
labse,0.46,0.48,0.38
minilm,0.64,0.42,0.46
tfidf,0.72,0.06,0.64


In [3]:
pd.pivot_table(pred, index="method", columns=["difficulty"], values="correct_top1", aggfunc="mean").round(2)

difficulty,hard,normal
method,,
hybrid,0.75,0.56
keyword,0.50,0.41
labse,0.38,0.45
minilm,0.54,0.50
tfidf,0.62,0.44


## 2. Where do errors go? Prediction frequency vs pack size

If one pack attracts many wrong predictions, check whether it is simply the pack with the most
(core) performance criteria: more criteria give more chances of a high maximum similarity.

In [4]:
sizes = {}
for f in (ROOT / "data/processed/qp").glob("*.json"):
    q = json.load(open(f))
    sizes[q["nqr_id"]] = sum(len(n["performance_criteria"]) for n in q["nos"] if n["core"] is not False)
wrong = pred[~pred["correct_top1"] & pred["pred_top1"].notna()]
tab = (wrong.groupby(["method", "pred_top1"]).size().unstack(fill_value=0).T)
tab.index = [f"{packs[int(i)][:28]} ({sizes[int(i)]} core PCs)" for i in tab.index]
tab

method,hybrid,keyword,labse,minilm,tfidf
Industrial Electrician (67 core PCs),28,15,58,37,15
Jr. Technician (Smart Energy (39 core PCs),6,2,7,7,2
Rooftop Solar PV (Installati (18 core PCs),2,7,1,1,7
Assistant Electrician (71 core PCs),19,10,17,25,5
Wireman Control Panel Electr (29 core PCs),6,5,1,4,3


## 3. Hindi (Devanagari): lexical baselines vs MiniLM

In [5]:
hi = pred[pred.language == "hi"].pivot_table(index="id", columns="method", values="correct_top1", aggfunc="first")
hi = hi.astype(bool)
print("Hindi items MiniLM gets right but TF-IDF does not:", int((hi["minilm"] & ~hi["tfidf"]).sum()))
print("Hindi items TF-IDF gets right but MiniLM does not:", int((~hi["minilm"] & hi["tfidf"]).sum()))
scores = pred[(pred.language == "hi") & (pred.method == "tfidf")]["scores"].map(lambda s: max(json.loads(s).values()))
print("Share of Hindi declarations where TF-IDF scores every pack 0:", round(float((scores == 0).mean()), 2))

Hindi items MiniLM gets right but TF-IDF does not: 19
Hindi items TF-IDF gets right but MiniLM does not: 1
Share of Hindi declarations where TF-IDF scores every pack 0: 0.94


## 4. The MiniLM errors, with text (for qualitative discussion)

In [6]:
pd.set_option("display.max_colwidth", 160)
err = pred[(pred.method == "minilm") & ~pred.correct_top1][["id", "language", "difficulty", "gold", "pred_top1", "text"]].copy()
err["gold"] = err["gold"].map(lambda i: packs[int(i)][:24])
err["pred_top1"] = err["pred_top1"].map(lambda i: packs[int(i)][:24] if pd.notna(i) else "(tie)")
err.head(25)

,id,language,difficulty,gold,pred_top1,text
304,S005,hinglish,normal,Wireman Control Panel El,Industrial Electrician,"Panel me component mounting, drilling, gland plate pe hole, phir wiring aur labeling. Faridabad me kaam karta hoon"
305,S006,hi,normal,Assistant Electrician,Wireman Control Panel El,"मैं इलेक्ट्रीशियन के साथ हेल्पर हूँ। दफ़्तर और दुकानों में कंसील्ड वायरिंग करते हैं। मैं पाइप मोड़ता हूँ, बॉक्स लगाता हूँ, तार डालता हूँ, फिर उस्ताद जी चेक ..."
307,S008,hi,normal,Industrial Electrician,Jr. Technician (Smart En,"फ़ैक्टरी में डीजी सेट चलाना और उसकी देखरेख मेरी ज़िम्मेदारी है। लाइट जाने पर चेंजओवर करता हूँ, डीज़ल, बैटरी, पानी सब रोज़ देखता हूँ।"
309,S010,hi,hard,Wireman Control Panel El,Industrial Electrician,"मैं पैनल वायरिंग करता हूँ। सेफ़्टी के लिए दस्ताने पहनता हूँ और कभी-कभी साइट पर जाकर पैनल में बाहर की केबल जोड़ने में मदद करता हूँ, पर मेरा मुख्य काम शॉप में..."
311,S012,hinglish,normal,Jr. Technician (Smart En,Industrial Electrician,"three phase meter dukaan aur atta chakki pe lagaya hai, phase sequence aur neutral dhyan se jodna padta hai warna reading galat aati hai"
312,S013,hinglish,normal,Assistant Electrician,Jr. Technician (Smart En,mai shuru me sirf saman uthata tha ab khud wiring kar leta hun. 2BHK ki puri wiring 4-5 din me ho jati hai do log ho to
313,S014,hi,normal,Rooftop Solar PV (Instal,Assistant Electrician,"पैनल पर धूल जम जाए या कोई कनेक्टर ढीला हो तो बिजली कम बनती है, मैं जाकर साफ़ करता हूँ और कनेक्शन टाइट करता हूँ।"
314,S015,hi,normal,Rooftop Solar PV (Instal,Assistant Electrician,"सोलर प्लेट की सफ़ाई और देखभाल का काम करता हूँ, हर महीने पानी और वाइपर से साफ़ करता हूँ।"
315,S016,en,normal,Wireman Control Panel El,Industrial Electrician,"I learnt from my senior in the panel workshop. At first I was only cutting wires and printing ferrules, now I wire full panels from the circuit diagram on m..."
318,S019,en,normal,Jr. Technician (Smart En,Industrial Electrician,"For shops and small workshops I install three phase meters, check the phase sequence and make sure the neutral is proper, then seal the terminal cover."


## 5. Observations (from the synthetic run on 28 Oct 2026; re-check them when re-running)

* Lexical baselines cannot match Devanagari text against English criteria: TF-IDF gives every pack a
  score of 0 for most Hindi declarations (section 3), so its Hindi accuracy is chance-level.
* MiniLM is the only single method that works in all three scripts, but it is weaker on romanised
  Hindi (Hinglish) than TF-IDF.
* Wrong predictions concentrate on *Industrial Electrician*. Pack size alone does not explain this
  (Assistant Electrician has more core criteria and attracts fewer errors); a likely cause is that
  its criteria use broad, generic wiring/testing/safety vocabulary. A length- or specificity-aware
  aggregation is a candidate fix, to be decided on the real dev set, not on this synthetic set.
* Contrary to our expectation, the "hard" declarations (which mention other trades) were **not**
  harder for most methods; with only 24 hard items this is weak evidence either way. One possible
  reason is that hard items were written with more detail.